In [163]:
import math
import tqdm
import string
import time
import random
import numpy as np
import pandas as pd
import re, os, random
import matplotlib.pyplot as plt

import pickle
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from torch.optim import Optimizer

import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

In [164]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [165]:
# Set device based on GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [166]:
torch.manual_seed(10701)
random.seed(10701)
np.random.seed(10701)

In [167]:
AA_VOCAB  = list('ACDEFGHIKLMNPQRSTVWXY')
SS8_VOCAB = ['C', 'B', 'E', 'G', 'I', 'H', 'S', 'T']
MAX_LEN   = 700

class ProteinDataset(Dataset):
    def __init__(self, df, max_len=MAX_LEN):
        self.df      = df.reset_index(drop=True)
        self.max_len = max_len
        self.aa_map  = {c: i for i, c in enumerate(AA_VOCAB)}
        self.ss_map  = {c: i for i, c in enumerate(SS8_VOCAB)}

    def __len__(self):
        return len(self.df)

    def encode(self, seq, vocab_map):
        arr = np.zeros(self.max_len, dtype=np.int64)
        for i, ch in enumerate(seq[:self.max_len]):
            if ch in vocab_map:
                arr[i] = vocab_map[ch]
        return arr

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        X      = self.encode(row['input'], self.aa_map)
        y      = self.encode(row['dssp8'], self.ss_map)
        length = min(len(row['input']), self.max_len)
        mask   = np.zeros(self.max_len, dtype=np.float32)
        mask[:length] = 1.0
        return (torch.tensor(X),
                torch.tensor(y),
                torch.tensor(mask))

In [168]:
train_df = pd.read_csv('/content/drive/MyDrive/2025-26/10701/train_data.csv')
test_df = pd.read_csv('/content/drive/MyDrive/2025-26/10701/test_data.csv')

train_dataset = ProteinDataset(train_df)
test_dataset  = ProteinDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

In [169]:
class GRU(nn.Module): # TODO: make bidirectional + add attention
    def __init__(self, vocab_size, input_size, hidden_size, output_size, batch_size, sigma=0.01):
        super().__init__()
        self.hidden_size = hidden_size
        self.batch_size = batch_size
        self.vocab_size = vocab_size
        self.output_size = output_size
        self.input_size = input_size
        self.sigma = sigma

        init_weight = lambda *shape: nn.Parameter(torch.randn(*shape) * sigma)
        triple = lambda: (init_weight(input_size, hidden_size),
                          init_weight(hidden_size, hidden_size),
                          nn.Parameter(torch.zeros(hidden_size)))
        self.W_xz, self.W_hz, self.b_z = triple()  # Update gate
        self.W_xr, self.W_hr, self.b_r = triple()  # Reset gate
        self.W_xh, self.W_hh, self.b_h = triple()  # Candidate hidden state


    def forward(self, inputs, H=None):
      inputs = inputs.transpose(0, 1) # (T, B, D) - transposed to loop over time, not batch
      if H is None:
          # Initial state with shape: (batch_size, hidden_size)
          B = inputs.size(1)
          H = torch.zeros(B, self.hidden_size, device=inputs.device)

      outputs = []
      for X in inputs:
          assert X.shape[0] == H.shape[0], f"X:{X.shape}, H:{H.shape}"
          Z = torch.sigmoid(torch.matmul(X, self.W_xz) +
                          torch.matmul(H, self.W_hz) + self.b_z)
          R = torch.sigmoid(torch.matmul(X, self.W_xr) +
                          torch.matmul(H, self.W_hr) + self.b_r)
          H_tilde = torch.tanh(torch.matmul(X, self.W_xh) +
                            torch.matmul(R * H, self.W_hh) + self.b_h)
          H = Z * H + (1 - Z) * H_tilde
          outputs.append(H)

      outputs = torch.stack(outputs, dim=1)
      return outputs, H


In [170]:
# https://pmc.ncbi.nlm.nih.gov/articles/PMC12221175/

class biGRU(nn.Module):
    def __init__(self, vocab_size, input_size, hidden_size, output_size, batch_size):
        super().__init__()

        self.hidden_size = hidden_size
        self.batch_size = batch_size
        self.vocab_size = vocab_size
        self.output_size = output_size
        self.input_size = input_size

        self.fwd_gru = GRU(vocab_size, input_size, hidden_size, output_size, batch_size) # Forward gru
        self.bwd_gru = GRU(vocab_size, input_size, hidden_size, output_size, batch_size) # Backward gru

        self.embedding = nn.Embedding(vocab_size, input_size)

    def forward(self, inputs, fwd_H=None, bwd_H=None):
        inputs = self.embedding(inputs)
        fwd_outputs, fwd_H = self.fwd_gru(inputs, fwd_H)

        bwd_inputs = torch.flip(inputs, dims=[1]) # reverse the inputs for backward gru
        bwd_outputs, bwd_H = self.bwd_gru(bwd_inputs, bwd_H)
        # outputs = torch.cat([fwd_outputs, bwd_outputs[::-1]], dim=-1) # un-reverse the outputs from backward gru

        bwd_outputs2 = torch.flip(bwd_outputs, dims=[1]) # un-reverse the outputs from backward gru

        outputs = []
        for h_f, h_b in zip(fwd_outputs, bwd_outputs2):
            outputs.append(torch.cat([h_f, h_b], dim=1))


        outputs = torch.stack(outputs, dim=1)
        H = torch.cat([fwd_H, bwd_H], dim=-1)
        return outputs, H


In [171]:
# https://link.springer.com/article/10.1186/s12859-025-06185-2#Sec2

class Adam_LM(Optimizer):
    def __init__(
        self,
        params,
        function,
        lr=1e-3,
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=0,
        amsgrad=False,
        eps_lm=1,
    ):
        defaults = dict(
            function=function,
            c=0.,
            running_loss=0.,
            lr=lr,
            betas=betas,
            eps=eps,
            weight_decay=weight_decay,
            amsgrad=amsgrad,
            eps_lm=eps_lm,
        )
        super(Adam_LM, self).__init__(params, defaults)

    def __setstate__(self, state):
        super(Adam_LM, self).__setstate__(state)
        for group in self.param_groups:
            group.setdefault("amsgrad", False)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            params_with_grad = []
            grads = []
            exp_avgs = []
            exp_avg_sqs = []
            max_exp_avg_sqs = []
            beta1, beta2 = group["betas"]
            eps_lm = group["eps_lm"]
            func = group["function"]
            c = group["c"]
            running_loss = group["running_loss"]

            for p in group["params"]:
                if p.grad is not None:
                    params_with_grad.append(p)
                    if p.grad.is_sparse:
                        raise RuntimeError(
                            "Adam does not support sparse gradients, please consider SparseAdam instead"
                        )

                    grad = p.grad
                    new_grad = grad / (
                        func(torch.max(torch.zeros(p.data.size(), device=p.device), running_loss - c))
                        + eps_lm
                    )
                    grads.append(new_grad)

                    state = self.state[p]
                    if len(state) == 0:
                        state["step"] = 0
                        state["exp_avg"] = torch.zeros_like(
                            p, memory_format=torch.preserve_format
                        ).to(p.device)
                        state["exp_avg_sq"] = torch.zeros_like(
                            p, memory_format=torch.preserve_format
                        ).to(p.device)
                        if group["amsgrad"]:
                            state["max_exp_avg_sq"] = torch.zeros_like(
                                p, memory_format=torch.preserve_format
                            ).to(p.device)

                    exp_avgs.append(state["exp_avg"])
                    exp_avg_sqs.append(state["exp_avg_sq"])

                    if group["amsgrad"]:
                        max_exp_avg_sqs.append(state["max_exp_avg_sq"])

                    state["step"] += 1

            self.adam(
                params_with_grad,
                grads,
                exp_avgs,
                exp_avg_sqs,
                max_exp_avg_sqs,
                amsgrad=group["amsgrad"],
                beta1=beta1,
                beta2=beta2,
                lr=group["lr"],
                weight_decay=group["weight_decay"],
                eps=group["eps"],
            )

        return loss

    def adam(
        self,
        params,
        grads,
        exp_avgs,
        exp_avg_sqs,
        max_exp_avg_sqs,
        *,
        amsgrad: bool,
        beta1: float,
        beta2: float,
        lr: float,
        weight_decay: float,
        eps: float,
    ):
        for i, param in enumerate(params):
            grad = grads[i]
            exp_avg = exp_avgs[i]
            exp_avg_sq = exp_avg_sqs[i]
            step = self.state[param]["step"]

            bias_correction1 = 1 - beta1 ** step
            bias_correction2 = 1 - beta2 ** step

            if weight_decay != 0:
                grad = grad.add(param, alpha=weight_decay)

            exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
            exp_avg_sq.mul_(beta2).addcmul_(grad, grad.conj(), value=1 - beta2)

            if amsgrad:
                torch.maximum(max_exp_avg_sqs[i], exp_avg_sq, out=max_exp_avg_sqs[i])
                denom = max_exp_avg_sqs[i].sqrt() / math.sqrt(bias_correction2).add_(eps)
            else:
                denom = exp_avg_sq.sqrt() / math.sqrt(bias_correction2).add_(eps)

            step_size = lr / bias_correction1

            param.addcdiv_(exp_avg, denom, value=-step_size)

In [172]:
def train(data_loader, model, criterion, optimizer, device):
    model.train()

    total_loss = 0
    total_tokens = 0
    num_correct = 0

    for samples, labels, mask in tqdm.tqdm(data_loader, leave=False):
        samples = samples.to(device)
        labels = labels.to(device)
        mask = mask.to(device)

        B = samples.size(0)
        # h0 = model.init_hidden(B, device)

        out, hidden = model(samples)

        out_flat = out.view(-1, out.shape[-1])
        labels_flat = labels.view(-1)
        mask_flat = mask.view(-1)

        loss = criterion(out_flat, labels_flat)
        loss = loss * mask_flat
        loss = loss.sum() / mask_flat.sum()

        group["running_loss"] = loss.item() # update running_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * mask_flat.sum().item()
        total_tokens += mask_flat.sum().item()

        preds = torch.argmax(out, dim=-1)
        num_correct += ((preds == labels) * mask).sum().item()

    return total_loss / total_tokens, num_correct / total_tokens

In [173]:
def test(data_loader, model, criterion, device):
    total_loss = 0
    total_tokens = 0
    num_correct = 0

    all_labels = []
    all_preds = []

    model.eval()

    with torch.no_grad():
        for samples, labels, mask in tqdm.tqdm(data_loader, leave=False):
            samples = samples.to(device)
            labels = labels.to(device)
            mask = mask.to(device).bool()

            B = samples.size(0)
            # h0 = model.init_hidden(B, device)

            out, hidden = model(samples)

            # flatten EVERYTHING first
            out_flat = out.view(-1, out.shape[-1])
            labels_flat = labels.view(-1)
            mask_flat = mask.view(-1)

            loss = criterion(out_flat, labels_flat)
            loss = loss * mask_flat
            loss = loss.sum() / mask_flat.sum()

            total_loss += loss.item() * mask_flat.sum().item()
            total_tokens += mask_flat.sum().item()

            preds = torch.argmax(out, dim=-1)

            preds_flat = preds.view(-1)
            labels_flat2 = labels.view(-1)

            # safe masking
            valid = mask_flat

            all_labels.extend(labels_flat2[valid].cpu().tolist())
            all_preds.extend(preds_flat[valid].cpu().tolist())

            num_correct += (preds_flat[valid] == labels_flat2[valid]).sum().item()

    return (
        total_loss / total_tokens,
        num_correct / total_tokens,
        all_labels,
        all_preds
    )

In [174]:
def run(num_epochs, train_dataloader, test_dataloader, rnn, criterion, optimizer):
    train_losses = []
    test_losses = []
    train_accs = []
    test_accs = []
    for epoch in range(num_epochs):
        print(f"Epoch {epoch}")
        # Perform one epoch of training and testing
        train_loss, train_acc = train(train_dataloader, rnn, criterion, optimizer, device)
        test_loss, test_acc = test(test_dataloader, rnn, criterion, device)

        train_losses.append(train_loss)
        test_losses.append(test_loss)
        train_accs.append(train_acc)
        test_accs.append(test_acc)

    return train_losses, test_losses, train_accs, test_accs

In [175]:
def plot_losses(epochs, train_losses, test_losses):
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_losses, label="Train", color="blue")
    plt.plot(epochs, test_losses, label="Validation", color="red")
    plt.title("Loss Over Epochs (biGRU)")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.savefig('/content/drive/MyDrive/2025-26/10701/biGRU_losses_plot.png')
    plt.close()

In [176]:
def plot_accs(epochs, train_accs, test_accs):
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_accs, label="Train", color="blue")
    plt.plot(epochs, test_accs, label="Validation", color="red")
    plt.title("Accuracy Over Epochs (biGRU)")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.savefig('/content/drive/MyDrive/2025-26/10701/biGRU_accs_plot.png')
    plt.close()

In [177]:
def plot_confusion_matrix(data_loader, model, title, as_percent = True, class_names=SS8_VOCAB):
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    all_preds = []
    all_labels = []

    model.eval()
    with torch.no_grad():
        for X, y, mask in tqdm.tqdm(data_loader, leave=False):
            X = X.to(device=device, dtype=torch.long)
            y = y.to(device=device, dtype=torch.long)
            mask = mask.to(device=device, dtype=torch.bool)

            B = X.size(0)
            h0 = model.init_hidden(B, device)

            y_hat = model.forward(X, h0)
            pred = y_hat.argmax(dim=-1).to(dtype=torch.long)

            all_preds.append(pred[mask].cpu())
            all_labels.append(y[mask].cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    fig, ax = plt.subplots(figsize=(10, 8))

    # cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(class_names))))
    cm = confusion_matrix(all_labels, all_preds)
    if as_percent:
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        sns.heatmap(cm_norm, annot=True, fmt=".1%", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
    else:
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)

    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.tight_layout()

    fig.savefig('/content/drive/MyDrive/2025-26/10701/biGRU_confusion_matrix.png', bbox_inches='tight', dpi=150)
    # plt.show()
    # print(classification_report(all_labels, all_preds, target_names=class_names))
    plt.close()

    return cm

In [178]:
def main(
    # Hyperparameters
    vocabulary_size = 20000,
    batch_size = 128,
    input_size = 64,
    hidden_size = 128,
    max_review_length = MAX_LEN,
    lr = 1e-4,
    num_epochs = 10,
    num_classes = 2
):
    input_size = MAX_LEN
    num_train_samples = 10790
    num_test_samples = 432
    AA_vocab_size = len(AA_VOCAB) # 21
    SS8_vocab_size = len(SS8_VOCAB) # 8
    num_classes = 8 # using SS8?

    # Initialize model
    model = biGRU(AA_vocab_size, MAX_LEN, hidden_size, num_classes, batch_size)
    model.to(device)

    # Initialize loss function and optimizer
    loss_fn = nn.CrossEntropyLoss(reduction='none')
    # optim = torch.optim.Adam(model.parameters(), lr)
    def identity(x):
      return x
    optim = Adam_LM(model.parameters(), identity, lr)

    # Run training and testing
    train_losses, test_losses, train_accs, test_accs = run(num_epochs, train_loader, test_loader, model, loss_fn, optim)
    # run(num_epochs, train_dataloader, test_dataloader, model, loss_fn, optim)

    # print(f"Train losses = {train_losses}")
    # print(f"Test losses = {test_losses}")

    plot_losses(list(range(num_epochs)), train_losses, test_losses)
    return train_losses, test_losses

In [179]:
main()

Epoch 0


TypeError: max() received an invalid combination of arguments - got (Tensor, float), but expected one of:
 * (Tensor input, *, Tensor out = None)
 * (Tensor input, Tensor other, *, Tensor out = None)
 * (Tensor input, int dim, bool keepdim = False, *, tuple of Tensors out = None)
 * (Tensor input, name dim, bool keepdim = False, *, tuple of Tensors out = None)
